In [ ]:
import torch
import numpy as np
import scanpy as sc
import scipy.sparse
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

class ScaledNonlinearFDTEngine:
    def __init__(self, latent_dim=50, device=None):
        self.latent_dim = latent_dim
        self.device = device if device else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.pca = PCA(n_components=latent_dim)
        self.scaler = StandardScaler()
        
        self.C2 = None 
        self.C3 = None 
        self.gene_loadings = None
        
    def fit_control_fluctuations(self, control_expression):
        print("Scaling control data to z-scores (mu=0, sigma=1)...")
        # This is the critical fix: bounds the math so the Taylor expansion doesn't explode
        scaled_control = self.scaler.fit_transform(control_expression)
        
        print(f"Running PCA to reduce down to {self.latent_dim} dimensions...")
        latent_Z = self.pca.fit_transform(scaled_control)
        self.gene_loadings = torch.tensor(self.pca.components_, dtype=torch.float32, device=self.device)
        
        Z_tensor = torch.tensor(latent_Z, dtype=torch.float32, device=self.device)
        Z_centered = Z_tensor - Z_tensor.mean(dim=0)
        N = Z_centered.shape[0]
        
        print("Computing 2-point Covariance Matrix (Linear Baseline)...")
        self.C2 = torch.matmul(Z_centered.T, Z_centered) / (N - 1)
        
        print("Computing 3-point Covariance Tensor (Nonlinear FDT)...")
        self.C3 = torch.einsum('ni,nj,nk->ijk', Z_centered, Z_centered, Z_centered) / N
        print("FDT Tensors computed on GPU.")
        
        return scaled_control

    def map_knockdown_to_field(self, target_gene_idx, scaled_kd_strength):
        # We now map the field in the strictly scaled space
        gene_vector = self.gene_loadings[:, target_gene_idx]
        u_latent = scaled_kd_strength * gene_vector
        return u_latent

    def predict_response(self, u_latent):
        dx_linear = torch.matmul(self.C2, u_latent)
        dx_quadratic = 0.5 * torch.einsum('ijk,j,k->i', self.C3, u_latent, u_latent)
        dx_nonlinear = dx_linear + dx_quadratic
        return dx_linear, dx_nonlinear

    def decode_to_scaled_genes(self, latent_dx):
        latent_dx_cpu = latent_dx.detach().cpu().numpy()
        return self.pca.inverse_transform(latent_dx_cpu)

def run_scaled_falsification():
    engine = ScaledNonlinearFDTEngine(latent_dim=50)
    file_path = "/kaggle/input/datasets/ericdu847/v5g0-vcc-00-probe/adata_Training.h5ad"
    print(f"Loading VCC dataset...")
    adata = sc.read_h5ad(file_path)
    
    pert_col = 'target_gene'
    control_name = 'non-targeting'
    
    control_mask = adata.obs[pert_col] == control_name
    control_adata = adata[control_mask]
    control_data = control_adata.X.toarray() if scipy.sparse.issparse(control_adata.X) else control_adata.X
        
    engine.fit_control_fluctuations(control_data)
    
    pert_counts = adata.obs[pert_col].value_counts()
    test_gene_name = pert_counts[pert_counts.index != control_name].index[0]
    target_gene_idx = list(adata.var_names).index(test_gene_name)
    
    pert_adata = adata[adata.obs[pert_col] == test_gene_name]
    pert_data_raw = pert_adata.X.toarray() if scipy.sparse.issparse(pert_adata.X) else pert_adata.X
    
    # Scale the perturbation data using the control cell scaler
    pert_mean_scaled = engine.scaler.transform(pert_data_raw.mean(axis=0).reshape(1, -1))[0]
    control_mean_scaled = np.zeros(adata.n_vars) # By definition of StandardScaler
    
    true_dx_scaled = pert_mean_scaled - control_mean_scaled
    actual_kd_strength_scaled = true_dx_scaled[target_gene_idx]
    
    print(f"\nTarget: {test_gene_name} (Index: {target_gene_idx})")
    print(f"Scaled knockdown strength (in Standard Deviations): {actual_kd_strength_scaled:.4f}")
    
    u_field = engine.map_knockdown_to_field(target_gene_idx, actual_kd_strength_scaled)
    dx_lin_latent, dx_nonlin_latent = engine.predict_response(u_field)
    
    dx_lin_genes_scaled = engine.decode_to_scaled_genes(dx_lin_latent)
    dx_nonlin_genes_scaled = engine.decode_to_scaled_genes(dx_nonlin_latent)
    
    mask = np.ones(adata.n_vars, dtype=bool)
    mask[target_gene_idx] = False
    
    mse_linear = mean_squared_error(true_dx_scaled[mask], dx_lin_genes_scaled[mask])
    mse_nonlinear = mean_squared_error(true_dx_scaled[mask], dx_nonlin_genes_scaled[mask])
    
    print("\n=== FALSIFICATION RESULTS (SCALED SPACE) ===")
    print(f"Linear FDT MSE:    {mse_linear:.6f}")
    print(f"Nonlinear FDT MSE: {mse_nonlinear:.6f}")

if __name__ == "__main__":
    run_scaled_falsification()